# exp77: N2, a stronger distilled fine-tune of the seed314159 secondary (TRAINING ONLY, never submit)

This notebook only trains. It writes no submission file, and it must never be submitted to the competition.

It runs **two recipes in parallel** in one T4 x2 session, as two subprocesses (one per GPU, `num_workers=2` each):

| run | GPU | init | lr | loader seed | length |
|---|---|---|---|---|---|
| **n2a** | `CUDA_VISIBLE_DEVICES=0` | exp71 student `ed8c616b` (private dataset `biohub-secft-all-v1`) | 1e-5 (same as exp71) | 271828 | 1 more epoch (epoch 2) |
| **n2b** | `CUDA_VISIBLE_DEVICES=1` | pilkwang parent `9bac2fa0` | **3e-5** (3x) | 314159 (same as exp71) | 1 epoch |

**Recipe (both runs):** exp71 = y3uanm `notebook0afc5663f6` (v1).
- Teacher: the frozen ORIGINAL pilkwang secondary `9bac2fa0` (also for n2a).
- Trainable: `unet.decoder_blocks[-1]`, `unet.head` and `detect_head`. BatchNorm stays frozen.
- Detection loss: softplus at the GT centres, plus 0.5 x BCE to the teacher on non-GT voxels, plus 0.1 x feature MSE to the teacher.
- AdamW with a fresh optimizer state, batch 1, fp32, all 199 annotated videos, no validation, no augmentations.

**Cells:**
- c01-c03: exp71's cells, byte for byte (setup + trainer patch, read check, splits + parent pin).
- c04: pins the exp71 student.
- The `%%writefile` cell: the worker. It holds exp71's c04-c07 recipe and run control, with the changes marked `[exp77]`.
- c06: the launcher.
- c07: the per-student sanity (hard asserts).
- c08: the per-student export and cleanup.

**Safety, per run (kept from exp71):**
- a 40-iteration timing probe;
- `max_iters` of at most one pass over the loader;
- a 300 s DataLoader timeout, and a retry with `num_workers=0` if the probe's loader fails;
- if the real run's loader fails, the student is saved.

**Time caps:**
- Each run's hard stop is 4.5 h of training. The student is saved.
- All training must end within 5.75 h of c04.
- The launcher watchdog touches a stop file 5 min after a run's deadline, then kills the process group 20 min after it.
- If only one GPU is visible, the runs go one after the other and share the window.

**Output:**
- `/kaggle/working/secft_n2a/` and `/kaggle/working/secft_n2b/`, each with `ARTIFACT_MANIFEST.json` (`model.weight_sha256`, `parent_sha256`, `root_parent_sha256`) and `weights/unet_transformer/split_0/{edge_predictor_best.pth, config.json}`;
- `/kaggle/working/exp77_train_log.json`;
- `/kaggle/working/exp77_logs/`.


In [ ]:
import copy
import hashlib
import json
import shutil
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import copy
import torch
import torch.nn.functional as functional

import time

input_root = Path("/kaggle/input")

# 新模型包上传到 Kaggle 后，修改这一处即可。
# 第一次使用原始模型时改成：
# SECONDARY_SLUG = "biohub-temporal-unet3d-seed314159-v1"
SECONDARY_SLUG = "biohub-temporal-unet3d-seed314159-v1"


def choose_existing_dir(name, candidates):
    for path in candidates:
        if path.is_dir():
            print(f"{name}: {path}", flush=True)
            return path
    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"{name} not found. Checked:\n{checked}"
    )


path_t0 = time.perf_counter()

support = choose_existing_dir(
    "support",
    [
        input_root / "datasets" / "pilkwang"
        / "biohub-tracking-support-pack-50ep-v1",
        input_root / "datasets" / "hftams"
        / "biohub-tracking-support-pack-50ep-v1",
        input_root / "biohub-tracking-support-pack-50ep-v1",
    ],
)

secondary = choose_existing_dir(
    "secondary",
    [
        input_root / "datasets" / "pilkwang" / SECONDARY_SLUG,
        input_root / "datasets" / "hftams" / SECONDARY_SLUG,
        input_root / SECONDARY_SLUG,
    ],
)

competition = choose_existing_dir(
    "competition",
    [
        input_root / "competitions"
        / "biohub-cell-tracking-during-development",
        input_root / "biohub-cell-tracking-during-development",
    ],
)

assert (competition / "train").is_dir(), (
    f"train directory not found: {competition / 'train'}"
)

print(
    f"Path initialization: {time.perf_counter() - path_t0:.2f}s",
    flush=True,
)

run_dir = Path(tempfile.mkdtemp(prefix="secondary_ft_", dir="/kaggle/working"))
repo_dir = run_dir / "repo"
initial_weights_root = run_dir / "initial_weights"

print("support:", support)
print("secondary:", secondary)
print("competition:", competition)
print("run_dir:", run_dir)

# 只安装训练脚本实际需要的顶层包。
# --no-deps 是关键：避免 tracksdata 自动拉取 imagecodecs==2026.6.26。
runtime_packages = [
    "tracksdata",
    "polars==1.42.0",
    "polars-runtime-32==1.42.0",
    "zarr==3.2.1",
    "geff==1.2.0.1.1",
    "geff-spec==1.1.1",
    "numcodecs==0.15.1",
    "donfig==0.8.1.post1",
    "bidict==0.23.1",
    "ilpy==0.6.0",
    "pyscipopt==6.2.1",
    "dask",
    "blosc2",
    "pyarrow",
    "rustworkx",
    "sqlalchemy",
]

wheels_dir = support / "wheels"
assert wheels_dir.is_dir(), f"wheels directory not found: {wheels_dir}"

install_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-index",
    "--no-deps",
    "--force-reinstall",
    "--find-links",
    str(wheels_dir),
    *runtime_packages,
]

print("Installing runtime packages with --no-deps...")
print(" ".join(install_command))
subprocess.run(install_command, check=True)

# 复制推理仓库源码。
repo_source = support / "repo"
repo_archive = support / "repo.zip"

if repo_source.is_dir():
    shutil.copytree(repo_source, repo_dir)
elif repo_archive.is_file():
    shutil.unpack_archive(str(repo_archive), str(run_dir))
    if not repo_dir.exists():
        extracted = next(
            path for path in run_dir.iterdir()
            if path.is_dir() and path.name not in {"initial_weights"}
        )
        extracted.rename(repo_dir)
else:
    raise FileNotFoundError("Support pack contains neither repo/ nor repo.zip")

# 复制副模型权重。
secondary_weights_source = secondary / "weights"
secondary_weights_archive = secondary / "weights.zip"

if secondary_weights_source.is_dir():
    shutil.copytree(secondary_weights_source, initial_weights_root)
elif secondary_weights_archive.is_file():
    shutil.unpack_archive(str(secondary_weights_archive), str(initial_weights_root))
else:
    raise FileNotFoundError("Secondary artifact contains neither weights/ nor weights.zip")

secondary_weights = (
    initial_weights_root
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
secondary_config = secondary_weights.with_name("config.json")

assert secondary_weights.is_file(), secondary_weights
assert secondary_config.is_file(), secondary_config

sys.path[:0] = [
    str(repo_dir / "scripts"),
    str(repo_dir / "src"),
]

train_script = repo_dir / "scripts" / "train_unet_transformer.py"
# [exp71] guard (not in y3uanm v1): patch only the exact support-pack trainer that x138 pins.
assert hashlib.sha256(train_script.read_bytes()).hexdigest() == (
    "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea"
), "support-pack train_unet_transformer.py is not the pinned version"
train_source = train_script.read_text(encoding="utf-8")


def replace_once(source, old, new, label):
    count = source.count(old)
    assert count == 1, (
        f"{label}: expected exactly one match, found {count}"
    )
    return source.replace(old, new, 1)


# 1. 给 train() 增加 validate 参数。
train_source = replace_once(
    train_source,
    """    pool_kernel_um: float = 5.0,
    data_parallel: bool = True,
) -> UNetNodeTransformer:""",
    """    pool_kernel_um: float = 5.0,
    data_parallel: bool = True,
    validate: bool = True,
) -> UNetNodeTransformer:""",
    "train signature",
)

# 3. validate=False 时不加载验证数据。
train_source = replace_once(
    train_source,
    """    train_video_data = _load(train_files, "train")
    test_video_data = _load(test_files, "test")
""",
    """    train_video_data = _load(train_files, "train")
    test_video_data = _load(test_files, "test") if validate else []
""",
    "validation data loading",
)


# 4. 不创建验证 Dataset。
train_source = replace_once(
    train_source,
    "    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes)\n",
    "    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes) if validate else None\n",
    "validation dataset",
)


# 5. 不创建验证 DataLoader。
train_source = replace_once(
    train_source,
    """    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0, pin_memory=False,
        generator=g, worker_init_fn=worker_init_fn,
    )
""",
    """    test_loader = (
        DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            prefetch_factor=2 if num_workers > 0 else None,
            persistent_workers=num_workers > 0,
            pin_memory=False,
            generator=g,
            worker_init_fn=worker_init_fn,
        )
        if validate
        else None
    )
""",
    "validation dataloader",
)


# 6. validate=False 时跳过完整 evaluate()。
train_source = replace_once(
    train_source,
    """        t0 = time.monotonic()
        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)
        test_time = time.monotonic() - t0

        score = test_acc * test_recall
        is_best = score >= best_score

        if is_best:
            best_score = score
            # Normalise any DataParallel "unet.module." prefix to "unet." so the
            # checkpoint loads on a single GPU (e.g. in the prediction script).
            torch.save(
                {k.replace("unet.module.", "unet.", 1): v for k, v in model.state_dict().items()},
                save_path,
            )
""",
    """        if validate:
            t0 = time.monotonic()
            test_loss, test_acc, test_recall = evaluate(
                model,
                test_loader,
                device,
                pool_kernel_um=pool_kernel_um,
            )
            test_time = time.monotonic() - t0

            score = test_acc * test_recall
            is_best = score >= best_score
        else:
            test_loss = float("nan")
            test_acc = float("nan")
            test_recall = float("nan")
            test_time = 0.0
            score = 0.0
            is_best = True

        if is_best:
            best_score = score
            # Normalize any DataParallel prefix before saving.
            torch.save(
                {
                    k.replace("unet.module.", "unet.", 1): v
                    for k, v in model.state_dict().items()
                },
                save_path,
            )
""",
    "evaluation block",
)


# 7. 训练结束时输出正确的模式说明。
train_source = replace_once(
    train_source,
    """    print(f"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}", flush=True)
""",
    """    if validate:
        print(
            f"\\nBest score (acc*recall): {best_score:.4f}, "
            f"saved to {save_path}",
            flush=True,
        )
    else:
        print(
            f"\\nTrain-only mode: latest model saved to {save_path}",
            flush=True,
        )
""",
    "training summary",
)


compile(train_source, str(train_script), "exec")
train_script.write_text(train_source, encoding="utf-8")

import inspect
import train_unet_transformer as trainer

assert Path(trainer.__file__).resolve() == train_script.resolve(), (
    f"Loaded wrong trainer: {trainer.__file__}"
)
assert "validate" in inspect.signature(trainer.train).parameters, (
    "Loaded trainer has no validate parameter. Restart the kernel."
)
assert torch.cuda.is_available(), "CUDA GPU is required"

print("Training script patched and imported.", flush=True)
print("trainer:", trainer.__file__, flush=True)
print("train signature:", inspect.signature(trainer.train), flush=True)
print("torch:", torch.__version__, flush=True)
print("CUDA:", torch.cuda.get_device_name(0), flush=True)
print("secondary weights:", secondary_weights, flush=True)
print("secondary config:", secondary_config, flush=True)


In [ ]:
import numpy as np
import zarr

train_root = competition / "train"
sample = next(
    path for path in sorted(train_root.glob("*.zarr"))
    if path.with_suffix(".geff").exists()
)
dataset = trainer.open_dataset(
    sample, require_tracks=True, load_image=False, downsample=(1, 4, 4)
)
image = zarr.open_group(str(dataset.zarr_path), mode="r")["0"]
frame = np.asarray(image[0, :, ::4, ::4])
assert frame.ndim == 3 and frame.size > 0 and np.isfinite(frame).all()
print("读取成功:", sample.name, "图像:", frame.shape)
print("标注节点数:", len(dataset.tracks.node_attrs()))


In [ ]:
initial_weights = secondary_weights
actual_initial_sha256 = hashlib.sha256(initial_weights.read_bytes()).hexdigest()
expected_initial_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
assert actual_initial_sha256 == expected_initial_sha256, "Initial weight SHA256 mismatch"
config = json.loads(initial_weights.with_name("config.json").read_text(encoding="utf-8"))
initial_state = torch.load(initial_weights, map_location="cpu", weights_only=True)
data_dir = competition / "train"
names = sorted(
    path.stem for path in data_dir.glob("*.zarr")
    if path.with_suffix(".geff").exists()
)
assert names, f"No annotated training datasets found in {data_dir}"
split = {"split": 0, "train": names, "test": []}
splits_path = run_dir / "splits.json"
splits_path.write_text(json.dumps([split], indent=2), encoding="utf-8")
print(f"Train: {len(names)}; local validation disabled", flush=True)


In [ ]:
# [exp77] Inputs beyond exp71. Cells c01-c03 above are exp71's, byte for byte. They install the wheels, copy the repo,
# patch the trainer and pin the pilkwang parent/teacher 9bac2fa0. They also write splits.json with all 199 videos and no test.
# This cell adds three things:
#   1. the exp71 student (ed8c616b, private dataset vitaliineibauer/biohub-secft-all-v1), the init of run A (n2a);
#   2. the sha of the PATCHED trainer file, so each worker subprocess can check it imports the same code;
#   3. the notebook clock that the launcher budgets against.
EXP77_T0 = time.time()
WORKING_ROOT = Path("/kaggle/working")
SECFT_ALL_SHA256 = "ed8c616bcc5990d3cccfe728d0e00469feb40ca8a0590137045f7ef225903f2e"

secft_all = choose_existing_dir(
    "secft_all (exp71 student)",
    [
        input_root / "datasets" / "vitaliineibauer" / "biohub-secft-all-v1",
        input_root / "biohub-secft-all-v1",
    ],
)
secft_all_manifest = json.loads((secft_all / "ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
assert secft_all_manifest["model"]["weight_sha256"] == SECFT_ALL_SHA256, secft_all_manifest["model"]["weight_sha256"]
assert secft_all_manifest["model"]["parent_sha256"] == expected_initial_sha256, secft_all_manifest["model"]["parent_sha256"]
secft_all_weights = secft_all / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
assert hashlib.sha256(secft_all_weights.read_bytes()).hexdigest() == SECFT_ALL_SHA256, "exp71 student SHA256 mismatch"
assert json.loads(secft_all_weights.with_name("config.json").read_text(encoding="utf-8")) == config, (
    "exp71 student config differs from the parent config"
)
secft_all_state = torch.load(secft_all_weights, map_location="cpu", weights_only=True)
assert list(secft_all_state) == list(initial_state), "exp71 student keys differ from the parent"
_moved = [k for k in initial_state if not torch.equal(secft_all_state[k], initial_state[k])]
assert _moved, "the exp71 student equals the parent"
print(f"exp71 student: {secft_all_weights} | sha {SECFT_ALL_SHA256[:8]} | {len(_moved)} tensors differ from the parent",
      flush=True)

PATCHED_TRAIN_SCRIPT_SHA256 = hashlib.sha256(train_script.read_bytes()).hexdigest()
print("patched trainer sha256:", PATCHED_TRAIN_SCRIPT_SHA256, flush=True)
print("visible GPUs in the notebook process:", torch.cuda.device_count(), flush=True)


In [ ]:
%%writefile /kaggle/working/exp77_worker.py
# [exp77] WORKER: runs ONE recipe of exp77 on ONE GPU.
# The notebook writes this file to /kaggle/working/exp77_worker.py (build.py prepends %%writefile). The launcher cell
# then starts it once per recipe as `python -u exp77_worker.py <run_cfg.json>`, with CUDA_VISIBLE_DEVICES set to one GPU.
# The recipe code is exp71's cells c04-c07 (the y3uanm v1 recipe plus exp71's run control), kept verbatim where
# possible. Every change against exp71 is marked [exp77]:
#   - the teacher is always the ORIGINAL pilkwang parent (initial_state); the student starts from RUN_CFG["init_weights"];
#   - lr and the loader seed come from RUN_CFG;
#   - the knobs come from RUN_CFG; the hard stop also honours the launcher's absolute deadline and a stop file;
#   - .cuda() is .to(_DEVICE). This is identical on Kaggle; CPU is allowed only for the local smoke test;
#   - the student is written to <out_dir>/weights, then TRAIN_DONE.json, then a diagnostic detection proxy runs.
import copy
import datetime
import functools
import gc
import hashlib
import inspect
import json
import math
import multiprocessing
import os
import shutil
import statistics
import sys
import time
import traceback
from pathlib import Path

RUN_CFG = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
LABEL = RUN_CFG["label"]
OUT_DIR = Path(RUN_CFG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = Path(RUN_CFG["log_path"])
DONE_PATH = OUT_DIR / "TRAIN_DONE.json"
TAG = f"[exp77 {LABEL}]"

if sys.platform.startswith("linux"):
    # exp71 ran the trainer's DataLoader workers under fork, the Linux default before Python 3.14. Pin fork here:
    # the trainer's worker_init_fn is a local function, which cannot be pickled under spawn or forkserver.
    multiprocessing.set_start_method("fork", force=True)

sys.path[:0] = [RUN_CFG["repo_scripts"], RUN_CFG["repo_src"]]

import torch
import torch.nn.functional as functional

import train_unet_transformer as trainer


def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


assert Path(trainer.__file__).resolve() == Path(RUN_CFG["train_script"]).resolve(), f"Loaded wrong trainer: {trainer.__file__}"
assert _sha256(trainer.__file__) == RUN_CFG["patched_train_script_sha256"], "trainer differs from the notebook's patched copy"
assert "validate" in inspect.signature(trainer.train).parameters, "Loaded trainer has no validate parameter"
ALLOW_CPU = os.environ.get("EXP77_ALLOW_CPU") == "1"  # local CPU smoke test only; never set on Kaggle
if not ALLOW_CPU:
    assert torch.cuda.is_available(), "CUDA GPU is required"
    assert torch.cuda.device_count() == 1, (
        f"expected exactly one visible GPU, got {torch.cuda.device_count()} "
        f"(CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r})"
    )
_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{TAG} device {_DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}), "
      f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r}", flush=True)

# ---- inputs (exp71 c03, with the teacher and the student init split apart) ----
config = RUN_CFG["config"]
data_dir = Path(RUN_CFG["data_dir"])
splits_path = Path(RUN_CFG["splits_path"])
names = list(RUN_CFG["names"])
split = json.loads(splits_path.read_text(encoding="utf-8"))[0]
initial_weights = Path(RUN_CFG["teacher_weights"])  # the frozen teacher = pilkwang parent 9bac2fa0 (exp71 name kept)
expected_initial_sha256 = RUN_CFG["teacher_sha256"]
assert _sha256(initial_weights) == expected_initial_sha256, "Teacher weight SHA256 mismatch"
initial_state = torch.load(initial_weights, map_location="cpu", weights_only=True)
student_init_weights = Path(RUN_CFG["init_weights"])
student_init_sha256 = RUN_CFG["init_sha256"]
assert _sha256(student_init_weights) == student_init_sha256, "Student init weight SHA256 mismatch"
student_init_state = torch.load(student_init_weights, map_location="cpu", weights_only=True)
assert list(student_init_state) == list(initial_state), "student init keys differ from the teacher"
LR = float(RUN_CFG["lr"])
SEED = int(RUN_CFG["seed"])

# ---- exp71 c04 (y3uanm v1 model + loss), verbatim except the lines marked [exp77] ----
BaseModel = trainer.UNetNodeTransformer
reference = {}
class SecondaryFineTune(BaseModel):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.load_state_dict(initial_state, strict=True)
        reference["teacher"] = copy.deepcopy(self).to(_DEVICE).eval().requires_grad_(False)  # [exp77] .cuda() -> .to(_DEVICE)
        self.load_state_dict(student_init_state, strict=True)  # [exp77] student init: n2a = exp71 student, n2b = the parent itself
        self.requires_grad_(False)
        for module in (self.unet.decoder_blocks[-1], self.unet.head, self.detect_head): module.requires_grad_(True)
    def train(self, mode=True):
        return super().train(False)
    def encode(self, imgs):
        features, logits = super().encode(imgs)
        if torch.is_grad_enabled():
            with torch.no_grad(): teacher_features, teacher_logits = reference["teacher"].encode(imgs)
            reference["logits"] = iter(teacher_logits)
            reference["feature_loss"] = functional.mse_loss(features.float(), teacher_features.float()) / teacher_features.float().square().mean().clamp_min(1e-6)
        return features, logits
def sparse_detection_loss(logits, coords, mask, neg_weight=0.01):
    """结合标注中心监督、未标注区域蒸馏与共享特征约束。"""
    teacher_prob = next(reference["logits"]).float().sigmoid()
    points = coords[mask].long().clamp_min(0).minimum(torch.tensor(logits.shape[2:], device=logits.device) - 1)
    positive = torch.zeros_like(logits, dtype=torch.bool)
    positive[mask.nonzero()[:, 0], 0, points[:, 0], points[:, 1], points[:, 2]] = True
    positive_loss = functional.softplus(-logits.float())[positive].sum() / positive.sum().clamp_min(1)
    distillation = functional.binary_cross_entropy_with_logits(logits.float(), teacher_prob, reduction="none")
    return positive_loss + 0.5 * distillation[~positive].sum() / (~positive).sum().clamp_min(1) + 0.1 * reference["feature_loss"]

# ---- exp71 c05 (run control + instrumentation); the knobs come from RUN_CFG [exp77] ----
_K = RUN_CFG["knobs"]
PROBE_ITERS = int(_K["probe_iters"])
PROBE_WARMUP_SKIP = int(_K["probe_warmup_skip"])
TRAIN_BUDGET_S = float(_K["train_budget_s"])
HARD_STOP_S = float(_K["hard_stop_s"])
NUM_WORKERS = int(_K["num_workers"])
DATALOADER_TIMEOUT_S = float(_K["dataloader_timeout_s"])
PROGRESS_EVERY = int(_K["progress_every"])
EXPECTED_TRAIN_VIDEOS = int(_K["expected_train_videos"])
DEADLINE_UNIX = float(RUN_CFG["deadline_unix"])  # [exp77] absolute end of training set by the launcher
STOP_FILE = RUN_CFG["stop_file"]  # [exp77] the launcher touches this to stop training at the next batch
# Spec cap: 199 videos x 99 windows (T=100, W=2). The trainer skips windows with an empty GT frame, so the real
# loader is shorter (18,707 on Kaggle in exp71); the probe cell also caps at one pass.
EPOCH_ITERS = 19701

assert len(names) == EXPECTED_TRAIN_VIDEOS, (len(names), EXPECTED_TRAIN_VIDEOS)
assert split["train"] == names and split["test"] == [], "exp77 trains on ALL annotated videos"

RUN_LOG = {
    "kernel": "vitaliineibauer/biohub-exp77-secft-n2",
    "label": LABEL,
    "recipe": RUN_CFG["recipe"],
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "torch": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "n_visible_gpus": torch.cuda.device_count(),
    "train_videos": len(names),
    "teacher_sha256": expected_initial_sha256,
    "init_sha256": student_init_sha256,
    "init_desc": RUN_CFG["init_desc"],
    "knobs": {
        "probe_iters": PROBE_ITERS,
        "probe_warmup_skip": PROBE_WARMUP_SKIP,
        "train_budget_s": TRAIN_BUDGET_S,
        "hard_stop_s": HARD_STOP_S,
        "deadline_unix": DEADLINE_UNIX,
        "num_workers": NUM_WORKERS,
        "dataloader_timeout_s": DATALOADER_TIMEOUT_S,
        "epoch_iters": EPOCH_ITERS,
        "lr": LR,
        "batch_size": 1,
        "seed": SEED,
    },
    "runs": {},
}
_current_run = {}


def _write_log():
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOG_PATH.write_text(json.dumps(RUN_LOG, indent=2) + "\n", encoding="utf-8")


def _excepthook(exc_type, exc, tb):  # [exp77] an uncaught error still leaves a log behind
    RUN_LOG["error"] = "".join(traceback.format_exception(exc_type, exc, tb))[-4000:]
    try:
        _write_log()
    finally:
        sys.__excepthook__(exc_type, exc, tb)


sys.excepthook = _excepthook


class _WallClockStop(RuntimeError):
    """Raised from the batch iterator when the hard wall-clock stop of a run is reached."""


def _nocache_cycle(loader):
    """Drop-in for trainer._cycle.

    trainer._cycle is itertools.cycle, which keeps a reference to EVERY batch it yields (~0.55 MB per
    window -> ~10 GB of host RAM over 19,701 windows when max_iters is set). This version re-iterates
    the loader instead (same order on the first pass: same seeded sampler), and stamps each yielded
    batch for the timing probe, the progress lines and the hard wall-clock stop.
    """
    run = _current_run
    run["loader_len"] = len(loader)
    stamps = run["stamps"]
    while True:
        for batch in loader:
            now = time.monotonic()
            stop_at = run.get("hard_stop_at")
            if stop_at is not None and now > stop_at:
                RUN_LOG["runs"][run["label"]].update(hard_stopped=True, stop_reason="hard_wall_clock")
                raise _WallClockStop(
                    f"{run['label']}: hard wall-clock stop after {len(stamps)} iterations"
                )
            if run.get("stop_file") and os.path.exists(run["stop_file"]):  # [exp77] launcher watchdog
                RUN_LOG["runs"][run["label"]].update(hard_stopped=True, stop_reason="stop_file")
                raise _WallClockStop(
                    f"{run['label']}: stop file after {len(stamps)} iterations"
                )
            stamps.append(now)
            k = len(stamps)
            if PROGRESS_EVERY > 0 and k % PROGRESS_EVERY == 0:
                elapsed = now - stamps[0]
                spi = elapsed / max(k - 1, 1)
                target = run.get("max_iters") or k
                print(
                    f"{TAG} {run['label']}: it {k}/{target} | {elapsed / 60:.1f} min | "
                    f"{spi:.3f} s/it | eta {(target - k) * spi / 60:.1f} min",
                    flush=True,
                )
            yield batch


_orig_train_epoch = trainer.train_epoch
_orig_tqdm = trainer.tqdm
_orig_DataLoader = trainer.DataLoader


def _dataloader_with_timeout(*args, **kwargs):
    """trainer.DataLoader with a per-batch timeout when workers are used (the trainer passes timeout=0 = wait
    forever): a stuck worker then raises instead of silently burning the session up to the 12 h limit."""
    if kwargs.get("num_workers", 0) > 0 and DATALOADER_TIMEOUT_S > 0:
        kwargs.setdefault("timeout", DATALOADER_TIMEOUT_S)
    return _orig_DataLoader(*args, **kwargs)


def _is_loader_failure(exc):
    """DataLoader timeouts / dead workers surface as RuntimeError at next(batch_iter), before any compute."""
    return isinstance(exc, RuntimeError) and "DataLoader" in str(exc)


def _timed_train_epoch(model, *args, **kwargs):
    """Wraps trainer.train_epoch: keeps a handle on the student (for the hard stop) and logs the epoch."""
    _current_run["model"] = model
    t0 = time.monotonic()
    out = _orig_train_epoch(model, *args, **kwargs)
    stats = RUN_LOG["runs"][_current_run["label"]]
    stats["train_epoch_wall_s"] = round(time.monotonic() - t0, 1)
    stats["avg_edge_loss"] = float(out[0])
    stats["avg_det_loss"] = float(out[1])
    return out


trainer._cycle = _nocache_cycle
trainer.train_epoch = _timed_train_epoch
trainer.DataLoader = _dataloader_with_timeout
# The trainer's tqdm bars refresh every 0.1 s on stderr (~140k updates in a 4 h run); 60 s keeps the log small.
trainer.tqdm = functools.partial(_orig_tqdm, mininterval=60.0)


def _begin_run(label, max_iters, hard_stop_s=None, stop_file=None):
    _current_run.clear()
    _current_run.update(
        label=label,
        max_iters=max_iters,
        stamps=[],
        hard_stop_at=None if hard_stop_s is None else time.monotonic() + hard_stop_s,
        stop_file=stop_file,
    )
    RUN_LOG["runs"][label] = {"max_iters": max_iters, "hard_stopped": False, "stop_reason": None}


def _end_run(wall_s):
    """Summarise the per-iteration stamps of the current run; returns the per-iteration durations."""
    stamps = _current_run["stamps"]
    diffs = [b - a for a, b in zip(stamps, stamps[1:])]
    stats = RUN_LOG["runs"][_current_run["label"]]
    stats["n_iters"] = len(stamps)
    stats["wall_s_incl_data_loading"] = round(wall_s, 1)
    stats["loader_len"] = _current_run.get("loader_len")
    if diffs:
        stats["seconds_per_iter_mean"] = round(statistics.fmean(diffs), 4)
        stats["seconds_per_iter_by_1000"] = [
            round(statistics.fmean(diffs[i:i + 1000]), 4) for i in range(0, len(diffs), 1000)
        ]
    return diffs


print(
    f"{TAG} control: lr {LR:g}, seed {SEED}, init {student_init_sha256[:8]}, teacher {expected_initial_sha256[:8]}; "
    f"probe {PROBE_ITERS} it (skip {PROBE_WARMUP_SKIP}), budget {TRAIN_BUDGET_S / 3600:.2f} h, "
    f"hard stop {HARD_STOP_S / 3600:.2f} h (deadline in {(DEADLINE_UNIX - time.time()) / 3600:.2f} h), "
    f"num_workers {NUM_WORKERS} (loader timeout {DATALOADER_TIMEOUT_S:.0f} s), {len(names)} train videos",
    flush=True,
)
_write_log()


def _train_call(num_workers, max_iters, weights_root):
    """The v1 training call (exp71 c06/c07); only lr and seed come from RUN_CFG [exp77]."""
    trainer.UNetNodeTransformer = SecondaryFineTune
    trainer.compute_detection_loss = sparse_detection_loss
    trainer.WEIGHTS_PATH = weights_root
    return trainer.train(
        data_dir=data_dir,
        fold=0,
        splits_file=splits_path,
        method="unet_transformer",
        n_epochs=1,
        lr=LR,
        batch_size=1,
        num_workers=num_workers,
        seed=SEED,
        unet_out_channels=config["unet_out_channels"],
        unet_layers=config["unet_layers"],
        downsample=tuple(config["downsample"]),
        window_size=config["window_size"],
        det_loss_weight=1.0,
        max_iters=max_iters,
        validate=False,
        augmentations=None,
        data_parallel=False,
        pool_kernel_um=float(config.get("pool_kernel_um", 5.0)),
    )


def _free_cuda():
    reference.clear()
    _current_run.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---- exp71 c06: timing probe (throw-away weights root) ----
probe_weights_root = OUT_DIR / "probe_weights"


def _run_probe(num_workers):
    _begin_run("probe", PROBE_ITERS)
    RUN_LOG["runs"]["probe"]["num_workers"] = num_workers
    t0 = time.monotonic()
    probe = _train_call(num_workers, PROBE_ITERS, probe_weights_root)
    return probe, _end_run(time.monotonic() - t0)


_probe_failure = None
try:
    probe_model, probe_diffs = _run_probe(NUM_WORKERS)
except RuntimeError as exc:
    if NUM_WORKERS == 0 or not _is_loader_failure(exc):
        raise
    traceback.print_exc()
    _probe_failure = repr(exc)
if _probe_failure is not None:  # retried outside the except block so the failed loader/workers are released
    RUN_LOG["probe_worker_failure"] = _probe_failure
    print(f"{TAG} WARNING: DataLoader failure with num_workers={NUM_WORKERS}; retrying the probe with num_workers=0", flush=True)
    NUM_WORKERS = 0
    _free_cuda()
    probe_model, probe_diffs = _run_probe(NUM_WORKERS)
RUN_LOG["knobs"]["num_workers"] = NUM_WORKERS
assert len(probe_diffs) >= 2, f"timing probe produced too few iterations: {probe_diffs}"

probe_skip = min(PROBE_WARMUP_SKIP, len(probe_diffs) // 2)
probe_steady = probe_diffs[probe_skip:]
seconds_per_iter = statistics.fmean(probe_steady)
n_windows = int(_current_run["loader_len"])
# [exp77] the budget also respects the launcher's absolute deadline, minus a margin for the second data load and the
# save (it matters only for the 1-GPU sequential fallback; in parallel mode TRAIN_BUDGET_S binds)
_remaining_s = DEADLINE_UNIX - time.time()
_budget_s = min(TRAIN_BUDGET_S, _remaining_s - min(900.0, 0.1 * _remaining_s))
budget_iters = max(0, int(_budget_s / seconds_per_iter))
MAX_ITERS = min(EPOCH_ITERS, budget_iters)
if n_windows != EPOCH_ITERS:
    # Expected: 18,707 on the full train set (windows with an empty GT frame are skipped). Never exceed one
    # pass over the loader (= v1's single epoch).
    print(f"{TAG} NOTE: the loader has {n_windows} windows (cap {EPOCH_ITERS}); capping MAX_ITERS at one pass", flush=True)
    MAX_ITERS = min(MAX_ITERS, n_windows)
MAX_ITERS = max(1, MAX_ITERS)

RUN_LOG["runs"]["probe"].update(
    warmup_skipped=probe_skip,
    seconds_per_iter_steady=round(seconds_per_iter, 4),
    seconds_per_iter_steady_median=round(statistics.median(probe_steady), 4),
    seconds_per_iter_steady_max=round(max(probe_steady), 4),
    warmup_diffs_s=[round(d, 3) for d in probe_diffs[:probe_skip]],
)
RUN_LOG["max_iters_rule"] = {
    "formula": "min(19701, int(min(TRAIN_BUDGET_S, rem - min(900, 0.1*rem)) / seconds_per_iter)), rem = deadline - now; capped at the loader length",
    "train_budget_s": TRAIN_BUDGET_S,
    "budget_s_used": round(_budget_s, 1),
    "seconds_per_iter": round(seconds_per_iter, 4),
    "budget_iters": budget_iters,
    "epoch_iters": EPOCH_ITERS,
    "loader_len": n_windows,
    "max_iters": MAX_ITERS,
    "epoch_fraction": round(MAX_ITERS / n_windows, 4),
    "projected_train_s": round(MAX_ITERS * seconds_per_iter, 1),
}
print(
    f"{TAG} timing probe: {len(probe_diffs) + 1} it; steady {seconds_per_iter:.4f} s/it "
    f"(median {statistics.median(probe_steady):.4f}, max {max(probe_steady):.4f}, first {probe_skip} skipped); "
    f"probe wall incl. data loading {RUN_LOG['runs']['probe']['wall_s_incl_data_loading'] / 60:.1f} min",
    flush=True,
)
print(
    f"{TAG} MAX_ITERS = {MAX_ITERS}  (budget {_budget_s:.0f} s / {seconds_per_iter:.4f} = {budget_iters}); "
    f"{MAX_ITERS / n_windows:.1%} of one epoch; projected training {MAX_ITERS * seconds_per_iter / 3600:.2f} h",
    flush=True,
)

# Free the probe completely: student, teacher copy, last autograd graph, loader workers, weights on disk.
del probe_model
_free_cuda()
shutil.rmtree(probe_weights_root, ignore_errors=True)
if torch.cuda.is_available():
    print(f"{TAG} after probe cleanup: CUDA memory allocated {torch.cuda.memory_allocated() / 2**20:.0f} MiB", flush=True)
_write_log()

# ---- exp71 c07: the real run ----
# exp71 changes kept: num_workers=NUM_WORKERS; max_iters=MAX_ITERS from the probe; a hard wall-clock stop and a
# DataLoader-failure path that save the current student exactly like train() would.
# [exp77] The hard stop is min(HARD_STOP_S, the launcher's deadline) and it also honours the stop file.
_hard_stop_s = min(HARD_STOP_S, max(60.0, DEADLINE_UNIX - time.time()))
_begin_run("train", MAX_ITERS, hard_stop_s=_hard_stop_s, stop_file=STOP_FILE)
RUN_LOG["runs"]["train"]["hard_stop_s"] = round(_hard_stop_s, 1)
_t0 = time.monotonic()
weights_root = OUT_DIR / "weights"
try:
    model = _train_call(NUM_WORKERS, MAX_ITERS, weights_root)
except (_WallClockStop, RuntimeError) as stop:
    if not isinstance(stop, _WallClockStop):
        if not _is_loader_failure(stop) or "model" not in _current_run:
            raise
        traceback.print_exc()
        RUN_LOG["runs"]["train"].update(hard_stopped=True, stop_reason=f"dataloader_failure: {stop!r}")
    print(f"{TAG} WARNING: {stop!r}; saving the current student state the way train() does", flush=True)
    model = _current_run["model"]
    torch.save(
        {k.replace("unet.module.", "unet.", 1): v for k, v in model.state_dict().items()},
        trainer.WEIGHTS_PATH / "unet_transformer/split_0/edge_predictor_best.pth",
    )
_end_run(time.monotonic() - _t0)
_current_run.pop("model", None)
print(f"{TAG} real run: {RUN_LOG['runs']['train']}", flush=True)
candidate = trainer.WEIGHTS_PATH / "unet_transformer/split_0/edge_predictor_best.pth"
model.load_state_dict(torch.load(candidate, map_location="cpu", weights_only=True), strict=True)
candidate.with_name("config.json").write_text(json.dumps(config, indent=2))
candidate_sha256 = _sha256(candidate)
print(f"{TAG} Candidate: {candidate}\n{TAG} SHA256: {candidate_sha256}", flush=True)
RUN_LOG["candidate"] = {"path": str(candidate), "sha256": candidate_sha256}
_write_log()
# The launcher exports a student only if this marker exists and its sha matches the file.
DONE_PATH.write_text(json.dumps({
    "label": LABEL,
    "candidate": str(candidate),
    "sha256": candidate_sha256,
    "n_iters": RUN_LOG["runs"]["train"].get("n_iters"),
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}, indent=2), encoding="utf-8")


# ---- exp71 c08 (diagnostic part only). The hard-assert sanity runs in the notebook process. ----
# Diagnostic only; it never blocks the export. It counts peaks of the teacher, the student init (n2a only) and the
# student on 8 fixed TRAIN windows (in-sample), with the trainer's own peak finder (detect_and_match) at p >= 0.5 and
# p >= 0.965.
def _exp77_detection_proxy(models, n_per_prefix=4):
    thresholds = {"p0.5": 0.0, "p0.965": math.log(0.965 / 0.035)}
    picks = []
    for prefix in ("44b6", "6bba"):
        group = [n for n in names if n.startswith(prefix)]
        picks += group[:: max(1, len(group) // n_per_prefix)][:n_per_prefix]
    totals = {"gt": 0}
    rows = []
    for name in picks:
        video_meta, windows = trainer.load_dataset_windows(
            data_dir / name,
            window_size=config["window_size"],
            downsample=tuple(config["downsample"]),
        )
        item = trainer.FrameWindowDataset([(video_meta, windows)])[len(windows) // 2]
        imgs = item["imgs"].to(_DEVICE, dtype=torch.float32)[None]
        coords = item["coords"].to(_DEVICE)[None]
        masks = item["masks"].to(_DEVICE)[None]
        image_shape = tuple(int(v) for v in item["image_shape"].tolist())
        voxel_size = tuple(float(v) for v in item["voxel_size"].tolist())
        with torch.no_grad():
            logits_by = {who: m.encode(imgs)[1] for who, m in models.items()}
        row = {"video": name, "t_start": int(item["t_start"]), "gt": int(masks.sum())}
        totals["gt"] += row["gt"]
        for who, logits in logits_by.items():
            for tname, tlogit in thresholds.items():
                n_det = n_hit = 0
                for i in range(imgs.shape[1]):
                    _, _, det_mask, matches = trainer.detect_and_match(
                        logits[i], coords[:, i], masks[:, i], image_shape,
                        det_threshold=tlogit,
                        pool_kernel_um=float(config.get("pool_kernel_um", 5.0)),
                        voxel_size=voxel_size,
                        frame_index=i,
                        window_size=imgs.shape[1],
                    )
                    n_det += int(det_mask.sum())
                    n_hit += int((matches[0] >= 0).sum())
                row[f"{who}_{tname}_det"] = n_det
                row[f"{who}_{tname}_gt_hit"] = n_hit
                totals[f"{who}_{tname}_det"] = totals.get(f"{who}_{tname}_det", 0) + n_det
                totals[f"{who}_{tname}_gt_hit"] = totals.get(f"{who}_{tname}_gt_hit", 0) + n_hit
        rows.append(row)
    for tname in thresholds:
        t_det = totals[f"teacher_{tname}_det"]
        parts = []
        for who in models:
            if who == "teacher":
                continue
            w_det = totals[f"{who}_{tname}_det"]
            totals[f"{who}_over_teacher_{tname}_det"] = round(w_det / max(t_det, 1), 4)
            parts.append(f"{who} {w_det} (x{w_det / max(t_det, 1):.3f}, GT hit {totals[f'{who}_{tname}_gt_hit']}/{totals['gt']})")
        print(f"{TAG}   {tname}: teacher {t_det} (GT hit {totals[f'teacher_{tname}_gt_hit']}/{totals['gt']}) -> "
              + "; ".join(parts), flush=True)
    return {"note": "in-sample train windows; trainer peak finder, not the x138 pipeline", "totals": totals, "rows": rows}


try:
    _proxy_models = {"teacher": reference["teacher"]}
    if student_init_sha256 != expected_initial_sha256:  # n2a: also show where the student started (exp71)
        _init_model = BaseModel(
            unet=trainer.TemporalUNet3D(
                in_channels=1,
                out_channels=config["unet_out_channels"],
                layers=config["unet_layers"],
            ),
            unet_out_channels=config["unet_out_channels"],
            pos_feat_dim=4 * trainer._POS_EMBED_DIM,
        )
        _init_model.load_state_dict(student_init_state)
        _proxy_models["init"] = _init_model.to(_DEVICE).eval()
    _proxy_models["student"] = model.eval()
    print(f"{TAG} detection proxy ({' vs '.join(_proxy_models)}, 8 train windows):", flush=True)
    RUN_LOG["detection_proxy"] = _exp77_detection_proxy(_proxy_models)
except Exception as exc:  # diagnostic only
    traceback.print_exc()
    RUN_LOG["detection_proxy"] = {"error": repr(exc)}

RUN_LOG["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
_write_log()
print(f"{TAG} worker done", flush=True)


In [ ]:
# [exp77] LAUNCHER. It runs the two recipes as two subprocesses of the worker written by the cell above:
#   n2a: GPU0 (CUDA_VISIBLE_DEVICES=0), init = the exp71 student ed8c616b, lr 1e-5, loader seed 271828. This is epoch 2.
#   n2b: GPU1 (CUDA_VISIBLE_DEVICES=1), init = the pilkwang parent 9bac2fa0, lr 3e-5 (3x), loader seed 314159 (as exp71).
# Both runs use the exp71 loss against the frozen ORIGINAL teacher 9bac2fa0, frozen BN, the same trainable modules,
# num_workers 2 each (the box has 4 vCPUs) and at most one pass over the loader.
# Time caps:
#   - each worker stops training at min(its start + 4.5 h, the deadline set here) and saves the student;
#   - the launcher touches a stop file 5 min after that deadline, then kills the process group 20 min after it;
#   - all training must end by EXP77_T0 + 5.75 h, so the notebook finishes well under 7 h.
# If the session shows only ONE GPU, the two runs go one after the other on GPU0, and each gets half of the
# remaining window.
# The EXP77_* environment overrides exist only for the local CPU smoke test (as in exp71). On Kaggle the defaults apply.
import datetime
import math
import os
import signal

PROBE_ITERS = int(os.environ.get("EXP77_PROBE_ITERS", "40"))
PROBE_WARMUP_SKIP = int(os.environ.get("EXP77_PROBE_WARMUP_SKIP", "10"))
TRAIN_BUDGET_S = float(os.environ.get("EXP77_TRAIN_BUDGET_H", "4.25")) * 3600.0
HARD_STOP_S = float(os.environ.get("EXP77_HARD_STOP_H", "4.5")) * 3600.0
NUM_WORKERS = int(os.environ.get("EXP77_NUM_WORKERS", "2"))
DATALOADER_TIMEOUT_S = float(os.environ.get("EXP77_DATALOADER_TIMEOUT_S", "300"))
PROGRESS_EVERY = int(os.environ.get("EXP77_PROGRESS_EVERY", "500"))
EXPECTED_TRAIN_VIDEOS = int(os.environ.get("EXP77_EXPECTED_TRAIN_VIDEOS", "199"))
TRAIN_WINDOW_S = float(os.environ.get("EXP77_TRAIN_WINDOW_H", "5.75")) * 3600.0
STOP_GRACE_S = float(os.environ.get("EXP77_STOP_GRACE_S", "300"))
KILL_GRACE_S = float(os.environ.get("EXP77_KILL_GRACE_S", "1200"))
POLL_S = float(os.environ.get("EXP77_POLL_S", "60"))
MIN_RUN_WINDOW_S = float(os.environ.get("EXP77_MIN_RUN_WINDOW_S", "1800"))
TAIL_MAX_LINES = int(os.environ.get("EXP77_TAIL_MAX_LINES", "40"))
GPU_SLOTS = int(os.environ.get("EXP77_GPU_SLOTS", str(torch.cuda.device_count())))
assert GPU_SLOTS >= 1, "no GPU visible"
TRAIN_END_UNIX = EXP77_T0 + TRAIN_WINDOW_S

WORKER_SRC = WORKING_ROOT / "exp77_worker.py"  # written by the %%writefile cell
assert WORKER_SRC.is_file(), WORKER_SRC
EXP77_LOGS = WORKING_ROOT / "exp77_logs"
EXP77_LOGS.mkdir(parents=True, exist_ok=True)

RECIPE = ("y3uanm notebook0afc5663f6 (v1) teacher-distilled fine-tune of the pilkwang seed314159 secondary; "
          "teacher = frozen pilkwang parent 9bac2fa0")
RUN_SPECS = [
    {
        "label": "n2a",
        "lr": 1e-5,
        "seed": 271828,
        "init_weights": str(secft_all_weights),
        "init_sha256": SECFT_ALL_SHA256,
        "init_desc": "vitaliineibauer/biohub-secft-all-v1 (exp71 student: 1 epoch, lr 1e-5, seed 314159)",
        "export_dir": "secft_n2a",
        "artifact_name": "biohub-secft-n2a-v1",
    },
    {
        "label": "n2b",
        "lr": 3e-5,
        "seed": 314159,
        "init_weights": str(initial_weights),
        "init_sha256": expected_initial_sha256,
        "init_desc": "pilkwang/biohub-temporal-unet3d-seed314159-v1 (the parent = the teacher)",
        "export_dir": "secft_n2b",
        "artifact_name": "biohub-secft-n2b-v1",
    },
]
SPEC_BY_LABEL = {spec["label"]: spec for spec in RUN_SPECS}
SLOTS = [str(i) for i in range(min(GPU_SLOTS, len(RUN_SPECS)))]

EXP77_LOG = {
    "kernel": "vitaliineibauer/biohub-exp77-secft-n2",
    "recipe": RECIPE,
    "started_utc": datetime.datetime.fromtimestamp(EXP77_T0, datetime.timezone.utc).isoformat(timespec="seconds"),
    "torch": torch.__version__,
    "n_visible_gpus_notebook": torch.cuda.device_count(),
    "gpu_slots": SLOTS,
    "mode": "parallel" if len(SLOTS) >= len(RUN_SPECS) else "sequential",
    "train_videos": len(names),
    "teacher_sha256": expected_initial_sha256,
    "patched_train_script_sha256": PATCHED_TRAIN_SCRIPT_SHA256,
    "knobs": {
        "probe_iters": PROBE_ITERS,
        "probe_warmup_skip": PROBE_WARMUP_SKIP,
        "train_budget_s": TRAIN_BUDGET_S,
        "hard_stop_s": HARD_STOP_S,
        "num_workers": NUM_WORKERS,
        "dataloader_timeout_s": DATALOADER_TIMEOUT_S,
        "train_window_s": TRAIN_WINDOW_S,
        "stop_grace_s": STOP_GRACE_S,
        "kill_grace_s": KILL_GRACE_S,
    },
    "run_specs": RUN_SPECS,
    "launch": {},
}


def _run_cfg(spec, slot, deadline_unix):
    out_dir = run_dir / spec["label"]
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg = {
        "label": spec["label"],
        "recipe": RECIPE,
        "out_dir": str(out_dir),
        "log_path": str(EXP77_LOGS / f"{spec['label']}_log.json"),
        "stop_file": str(out_dir / "STOP"),
        "deadline_unix": deadline_unix,
        "repo_scripts": str(repo_dir / "scripts"),
        "repo_src": str(repo_dir / "src"),
        "train_script": str(train_script),
        "patched_train_script_sha256": PATCHED_TRAIN_SCRIPT_SHA256,
        "config": config,
        "data_dir": str(data_dir),
        "splits_path": str(splits_path),
        "names": names,
        "teacher_weights": str(initial_weights),
        "teacher_sha256": expected_initial_sha256,
        "init_weights": spec["init_weights"],
        "init_sha256": spec["init_sha256"],
        "init_desc": spec["init_desc"],
        "lr": spec["lr"],
        "seed": spec["seed"],
        "knobs": {
            "probe_iters": PROBE_ITERS,
            "probe_warmup_skip": PROBE_WARMUP_SKIP,
            "train_budget_s": TRAIN_BUDGET_S,
            "hard_stop_s": HARD_STOP_S,
            "num_workers": NUM_WORKERS,
            "dataloader_timeout_s": DATALOADER_TIMEOUT_S,
            "progress_every": PROGRESS_EVERY,
            "expected_train_videos": EXPECTED_TRAIN_VIDEOS,
        },
    }
    cfg_path = out_dir / "run_cfg.json"
    cfg_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    return cfg_path, cfg


def _launch(spec, slot, deadline_unix):
    cfg_path, cfg = _run_cfg(spec, slot, deadline_unix)
    env = dict(os.environ)
    env["CUDA_VISIBLE_DEVICES"] = slot
    env["PYTHONPATH"] = os.pathsep.join(
        p for p in (cfg["repo_scripts"], cfg["repo_src"], os.environ.get("PYTHONPATH", "")) if p
    )
    env["PYTHONUNBUFFERED"] = "1"
    env["OMP_NUM_THREADS"] = "2"  # 2 trainers + 4 loader workers share 4 vCPUs
    env["MKL_NUM_THREADS"] = "2"
    log_path = EXP77_LOGS / f"worker_{spec['label']}.log"
    log_file = open(log_path, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-u", str(WORKER_SRC), str(cfg_path)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        cwd=str(WORKING_ROOT),
        env=env,
        start_new_session=True,  # own process group: the watchdog can kill the loader workers too
    )
    state = {
        "spec": spec,
        "slot": slot,
        "proc": proc,
        "log_path": log_path,
        "log_file": log_file,
        "log_pos": 0,
        "stop_file": Path(cfg["stop_file"]),
        "deadline_unix": deadline_unix,
        "launched_unix": time.time(),
        "stop_touched": False,
        "killed": False,
    }
    EXP77_LOG["launch"][spec["label"]] = {
        "slot_cuda_visible_devices": slot,
        "pid": proc.pid,
        "launched_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "deadline_utc": datetime.datetime.fromtimestamp(deadline_unix, datetime.timezone.utc).isoformat(timespec="seconds"),
        "lr": spec["lr"],
        "seed": spec["seed"],
        "init_sha256": spec["init_sha256"],
    }
    print(f"[exp77] launched {spec['label']} on CUDA_VISIBLE_DEVICES={slot} (pid {proc.pid}); lr {spec['lr']:g}, "
          f"seed {spec['seed']}, init {spec['init_sha256'][:8]}; training deadline in "
          f"{(deadline_unix - time.time()) / 3600:.2f} h", flush=True)
    return state


def _tail(state, final=False):
    """Print the new part of a worker log (tqdm \\r refreshes become lines), at most TAIL_MAX_LINES per poll."""
    with open(state["log_path"], "rb") as fh:
        fh.seek(state["log_pos"])
        data = fh.read()
        state["log_pos"] = fh.tell()
    if not data:
        return
    lines = [ln for ln in data.decode("utf-8", errors="replace").replace("\r", "\n").split("\n") if ln.strip()]
    limit = 10 * TAIL_MAX_LINES if final else TAIL_MAX_LINES
    if len(lines) > limit:
        print(f"[{state['spec']['label']}] ... {len(lines) - limit} lines skipped (full log: {state['log_path']})")
        lines = lines[-limit:]
    for ln in lines:
        print(f"[{state['spec']['label']}] {ln[:400]}")
    sys.stdout.flush()


def _kill(state):
    proc = state["proc"]
    try:
        if hasattr(os, "killpg"):
            os.killpg(proc.pid, signal.SIGKILL)
        else:
            proc.kill()
    except (ProcessLookupError, PermissionError, OSError) as exc:
        print(f"[exp77] kill {state['spec']['label']}: {exc!r}", flush=True)
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    state["killed"] = True


pending = list(RUN_SPECS)
active = {}
WORKER_RESULTS = {}
while pending or active:
    for slot in SLOTS:
        while slot not in active and pending:
            spec = pending.pop(0)
            now = time.time()
            # Parallel: 1 run per slot, it gets the whole window. Sequential fallback (1 GPU): the runs still to go
            # share the remaining window evenly.
            runs_to_go = math.ceil((1 + len(pending)) / len(SLOTS))
            share = (TRAIN_END_UNIX - now) / runs_to_go
            if share < MIN_RUN_WINDOW_S:
                print(f"[exp77] SKIP {spec['label']}: only {share / 60:.0f} min of training window left", flush=True)
                WORKER_RESULTS[spec["label"]] = {"returncode": None, "killed": False, "skipped": "no_time"}
                continue
            active[slot] = _launch(spec, slot, now + share)
    if not active:
        break
    time.sleep(POLL_S)
    for slot, state in list(active.items()):
        _tail(state)
        label = state["spec"]["label"]
        rc = state["proc"].poll()
        now = time.time()
        if rc is None and not state["stop_touched"] and now > state["deadline_unix"] + STOP_GRACE_S:
            state["stop_file"].touch()
            state["stop_touched"] = True
            print(f"[exp77] WATCHDOG: {label} still running {STOP_GRACE_S:.0f} s after its deadline; stop file touched",
                  flush=True)
        if rc is None and now > state["deadline_unix"] + KILL_GRACE_S:
            print(f"[exp77] WATCHDOG: killing {label} ({KILL_GRACE_S:.0f} s after its deadline)", flush=True)
            _kill(state)
            rc = state["proc"].poll()
        if rc is None:
            continue
        _tail(state, final=True)
        state["log_file"].close()
        WORKER_RESULTS[label] = {
            "returncode": rc,
            "killed": state["killed"],
            "stop_file_touched": state["stop_touched"],
            "wall_s": round(now - state["launched_unix"], 1),
        }
        print(f"[exp77] {label} exited with code {rc} after {(now - state['launched_unix']) / 3600:.2f} h"
              f"{' (KILLED by the watchdog)' if state['killed'] else ''}", flush=True)
        del active[slot]

# Collect the students: a run counts only if TRAIN_DONE.json exists and its sha matches the weights file.
STUDENTS = {}
for spec in RUN_SPECS:
    label = spec["label"]
    res = WORKER_RESULTS.setdefault(label, {"returncode": None, "killed": False})
    log_json = EXP77_LOGS / f"{label}_log.json"
    res["worker_log"] = json.loads(log_json.read_text(encoding="utf-8")) if log_json.is_file() else None
    done_path = run_dir / label / "TRAIN_DONE.json"
    if not done_path.is_file():
        res["status"] = "no_student"
        print(f"[exp77] {label}: NO STUDENT (TRAIN_DONE.json missing; returncode {res['returncode']})", flush=True)
        continue
    done = json.loads(done_path.read_text(encoding="utf-8"))
    cand = Path(done["candidate"])
    if not cand.is_file() or hashlib.sha256(cand.read_bytes()).hexdigest() != done["sha256"]:
        res["status"] = "bad_student_file"
        print(f"[exp77] {label}: candidate missing or sha mismatch vs TRAIN_DONE.json", flush=True)
        continue
    res["status"] = "trained"
    if res["returncode"] not in (0, None) or res["killed"]:
        print(f"[exp77] {label}: the student was saved, but the worker then failed or was killed "
              f"(returncode {res['returncode']}); exporting it anyway (the proxy may be missing)", flush=True)
    STUDENTS[label] = {"candidate": cand, "sha256": done["sha256"], "done": done}
EXP77_LOG["worker_results"] = {k: {kk: vv for kk, vv in v.items() if kk != "worker_log"} for k, v in WORKER_RESULTS.items()}
print(f"[exp77] students ready for sanity/export: {sorted(STUDENTS)}; "
      f"notebook elapsed {(time.time() - EXP77_T0) / 3600:.2f} h", flush=True)


In [ ]:
# [exp77] Sanity, the exp71 c08 hard asserts, run per student in the notebook process (CPU tensors). A student is exported
# only if every check passes:
#   - only parameters under the v1-trainable prefixes moved, against every reference: n2b vs the parent 9bac2fa0;
#     n2a vs its init ed8c616b AND vs the root parent 9bac2fa0;
#   - every tensor is finite;
#   - the sha differs from every reference;
#   - config.json equals the parent config;
#   - the file strict-loads the way predict_unet_transformer.load_model loads BIOHUB_SECONDARY_WEIGHTS in x138.
# A failing student is recorded and skipped, and the other one is still exported. The detection proxy ran in each worker
# and is in <label>_log.json.
import traceback

BaseModel = trainer.UNetNodeTransformer


def _build_base_model():
    return BaseModel(
        unet=trainer.TemporalUNet3D(
            in_channels=1,
            out_channels=config["unet_out_channels"],
            layers=config["unet_layers"],
        ),
        unet_out_channels=config["unet_out_channels"],
        pos_feat_dim=4 * trainer._POS_EMBED_DIM,
    )


_probe_model = _build_base_model()
_last_block = len(_probe_model.unet.decoder_blocks) - 1
del _probe_model
TRAINABLE_PREFIXES = (f"unet.decoder_blocks.{_last_block}.", "unet.head.", "detect_head.")
REFERENCES = {
    "n2a": {"init_ed8c616b": (secft_all_state, SECFT_ALL_SHA256), "root_parent_9bac2fa0": (initial_state, expected_initial_sha256)},
    "n2b": {"parent_9bac2fa0": (initial_state, expected_initial_sha256)},
}


def _compare(new_state, ref_state):
    changed, unchanged_trainable, param_deltas = [], [], {}
    for key, old in ref_state.items():
        new = new_state[key]
        assert new.shape == old.shape and new.dtype == old.dtype, f"shape/dtype changed: {key}"
        if new.is_floating_point():
            assert torch.isfinite(new).all(), f"non-finite values in {key}"
        if torch.equal(new, old):
            if key.startswith(TRAINABLE_PREFIXES):
                unchanged_trainable.append(key)
            continue
        changed.append(key)
        delta = new.double() - old.double()
        param_deltas[key] = {
            "max_abs": float(delta.abs().max()),
            "rel_l2": float(delta.norm() / old.double().norm().clamp_min(1e-12)),
        }
    outside = [key for key in changed if not key.startswith(TRAINABLE_PREFIXES)]
    assert not outside, f"parameters outside {TRAINABLE_PREFIXES} changed: {outside}"
    assert changed, "no parameter changed: the fine-tune did not update the model"
    unexpected_frozen = [
        key for key in unchanged_trainable
        if not key.endswith(("running_mean", "running_var", "num_batches_tracked"))
    ]
    return changed, unchanged_trainable, unexpected_frozen, param_deltas


def _sanity(label, student):
    candidate = student["candidate"]
    new_state = torch.load(candidate, map_location="cpu", weights_only=True)
    assert list(new_state) == list(initial_state), "state-dict keys differ from the teacher"
    candidate_sha256 = hashlib.sha256(candidate.read_bytes()).hexdigest()
    assert candidate_sha256 == student["sha256"], "candidate changed since TRAIN_DONE.json"
    out = {"candidate_sha256": candidate_sha256, "trainable_prefixes": list(TRAINABLE_PREFIXES), "vs": {}}
    for ref_name, (ref_state, ref_sha) in REFERENCES[label].items():
        assert candidate_sha256 != ref_sha, f"new weights are byte-identical to {ref_name}"
        changed, unchanged_trainable, unexpected_frozen, param_deltas = _compare(new_state, ref_state)
        if unexpected_frozen:
            print(f"WARNING [{label} vs {ref_name}]: trainable weights that did not move:", unexpected_frozen, flush=True)
        out["vs"][ref_name] = {
            "ref_sha256": ref_sha,
            "changed_keys": changed,
            "unchanged_trainable_keys": unchanged_trainable,
            "param_deltas": param_deltas,
        }
        print(f"[{label}] sanity vs {ref_name}: {len(changed)} tensors changed, all under {TRAINABLE_PREFIXES}")
        for key in changed:
            print(f"  {key:45s} max|d|={param_deltas[key]['max_abs']:.3e}  rel_l2={param_deltas[key]['rel_l2']:.3e}")
    _candidate_config = json.loads(candidate.with_name("config.json").read_text(encoding="utf-8"))
    assert _candidate_config == config, (_candidate_config, config)
    _strict_check = _build_base_model()
    _strict_check.load_state_dict(new_state)  # strict=True, same construction as load_model()
    del _strict_check
    out["all_finite"] = True
    out["strict_load"] = True
    print(f"[{label}] sanity OK: all finite; strict load OK; new SHA256 {candidate_sha256}", flush=True)
    return out


SANITY = {}
SANITY_FAIL = {}
for _label, _student in STUDENTS.items():
    try:
        SANITY[_label] = _sanity(_label, _student)
    except Exception as exc:
        traceback.print_exc()
        SANITY_FAIL[_label] = repr(exc)
        print(f"[exp77] {_label}: SANITY FAILED -> not exported: {exc!r}", flush=True)
EXP77_LOG["sanity"] = SANITY
EXP77_LOG["sanity_failures"] = SANITY_FAIL
if "n2a" in SANITY and "n2b" in SANITY and STUDENTS["n2a"]["sha256"] == STUDENTS["n2b"]["sha256"]:
    print("WARNING: n2a and n2b are byte-identical", flush=True)  # cannot happen with different inits; logged, not fatal
print(f"[exp77] sanity passed: {sorted(SANITY)}; failed: {SANITY_FAIL or 'none'}", flush=True)
for _label in sorted(SANITY):
    _proxy = (WORKER_RESULTS[_label].get("worker_log") or {}).get("detection_proxy", {})
    print(f"[{_label}] detection proxy totals: {_proxy.get('totals', _proxy)}", flush=True)


In [ ]:
# [exp77] LAST CELL: exp71 c09, run once per student that passed sanity. It writes the dataset payload in the layout of
# pilkwang/biohub-temporal-unet3d-seed314159-v1, which x138 code cell 3 reads:
#   - _find_secondary_artifact_root -> manifest["model"]["weight_sha256"];
#   - copy_or_extract_tree(<manifest dir>/weights) -> unet_transformer/split_0/{edge_predictor_best.pth, config.json};
#   - a sha256 check of the copied weights.
# It then removes every scratch dir from /kaggle/working.
#   /kaggle/working/secft_n2a/ARTIFACT_MANIFEST.json + weights/unet_transformer/split_0/{edge_predictor_best.pth, config.json}
#   /kaggle/working/secft_n2b/ (same layout)
#   /kaggle/working/exp77_train_log.json    (small combined log, outside the dataset folders)
#   /kaggle/working/exp77_logs/             (worker stdout logs, per-run JSON logs, the worker source)
teacher_manifest = json.loads((secondary / "ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
assert teacher_manifest["model"]["weight_sha256"] == expected_initial_sha256
compatibility = copy.deepcopy(teacher_manifest.get("compatibility", {}))
compatibility.pop("gate_diagnostics", None)  # refers to full_frame_center files this artifact does not ship
compatibility.get("models", {}).pop("full_frame_center", None)

PARENT_ARTIFACT = "pilkwang/biohub-temporal-unet3d-seed314159-v1"
EXPORTED = {}
EXPORT_FAIL = {}


def _export(label):
    spec = SPEC_BY_LABEL[label]
    candidate = STUDENTS[label]["candidate"]
    candidate_sha256 = SANITY[label]["candidate_sha256"]
    wlog = WORKER_RESULTS[label].get("worker_log") or {}
    train_stats = wlog.get("runs", {}).get("train", {})
    rule = wlog.get("max_iters_rule", {})
    out_dir = WORKING_ROOT / spec["export_dir"]
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_weights = out_dir / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
    out_weights.parent.mkdir(parents=True)
    shutil.copyfile(candidate, out_weights)
    # Architecture is unchanged: ship the teacher's own config.json bytes (same 5 keys load_model reads).
    shutil.copyfile(secondary_config, out_weights.with_name("config.json"))
    assert json.loads(out_weights.with_name("config.json").read_text(encoding="utf-8")) == config

    out_sha256 = hashlib.sha256(out_weights.read_bytes()).hexdigest()
    assert out_sha256 == candidate_sha256, (out_sha256, candidate_sha256)
    assert out_sha256 not in (expected_initial_sha256, SECFT_ALL_SHA256)

    n_iters = int(train_stats.get("n_iters") or STUDENTS[label]["done"].get("n_iters") or 0)
    is_continuation = spec["init_sha256"] != expected_initial_sha256
    lineage = (
        [
            {"artifact": "vitaliineibauer/biohub-secft-all-v1", "sha256": SECFT_ALL_SHA256,
             "training": "exp71: 1 epoch (18,707 it), lr 1e-5, seed 314159, same recipe"},
            {"artifact": PARENT_ARTIFACT, "sha256": expected_initial_sha256, "training": "pilkwang original"},
        ]
        if is_continuation
        else [{"artifact": PARENT_ARTIFACT, "sha256": expected_initial_sha256, "training": "pilkwang original"}]
    )
    model_block = {
        "config": config,
        "method": "unet_transformer",
        "weight_path": "weights/unet_transformer/split_0/edge_predictor_best.pth",
        "weight_bytes": out_weights.stat().st_size,
        "weight_sha256": out_sha256,
        "parent_sha256": spec["init_sha256"],
        "root_parent_sha256": expected_initial_sha256,
        "teacher_sha256": expected_initial_sha256,
        "lineage": lineage,
        "training": {
            "method": "secondary_teacher_distill_ft_y3uanm_v1",
            "exp77_run": label,
            "init": spec["init_desc"],
            "teacher": f"{PARENT_ARTIFACT} (frozen, sha {expected_initial_sha256[:8]})",
            "trainable": ["unet.decoder_blocks[-1]", "unet.head", "detect_head"],
            "batchnorm": "frozen (train() forced to eval mode)",
            "det_loss": "softplus(-logit) at GT centres + 0.5*BCE(student, teacher sigmoid) on non-GT voxels "
                        "+ 0.1*normalised feature MSE to the teacher",
            "edge_loss": "unchanged trainer edge loss through the frozen transformer",
            "det_loss_weight": 1.0,
            "optimizer": "AdamW (torch defaults besides lr; fresh optimizer state)",
            "learning_rate": spec["lr"],
            "batch_size": 1,
            "precision": "fp32",
            "seed": spec["seed"],
            "augmentations": [],
            "validation": None,
            "train_datasets": len(names),
            "loader_len": train_stats.get("loader_len"),
            "max_iters": train_stats.get("max_iters"),
            "n_iters": n_iters,
            "epoch_fraction": round(n_iters / max(int(train_stats.get("loader_len") or 0), 1), 4),
            "total_epochs_incl_parent_ft": (1.0 if is_continuation else 0.0)
                + round(n_iters / max(int(train_stats.get("loader_len") or 0), 1), 4),
            "hard_stopped": bool(train_stats.get("hard_stopped")),
            "stop_reason": train_stats.get("stop_reason"),
            "num_workers": (wlog.get("knobs") or {}).get("num_workers"),
            "cuda_visible_devices": wlog.get("cuda_visible_devices"),
            "seconds_per_iter_probe": rule.get("seconds_per_iter"),
            "seconds_per_iter_mean": train_stats.get("seconds_per_iter_mean"),
            "train_wall_s": train_stats.get("wall_s_incl_data_loading"),
        },
    }
    manifest = {
        "artifact_name": spec["artifact_name"],
        "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "source": f"private training kernel vitaliineibauer/biohub-exp77-secft-n2, run {label} "
                  "(y3uanm notebook0afc5663f6 recipe)",
        "parent_artifact": "vitaliineibauer/biohub-secft-all-v1" if is_continuation else PARENT_ARTIFACT,
        "root_parent_artifact": PARENT_ARTIFACT,
        "compatibility": compatibility,
        "contents": {
            "weights": {"files": 2, "bytes": sum(p.stat().st_size for p in out_weights.parent.iterdir())},
            "repo_included": False,
            "wheels_included": False,
        },
        "model": model_block,
        "models": {"unet_transformer": model_block},
        "n_iters": n_iters,
        "notes": [
            "Drop-in secondary for x138-style notebooks: same architecture/config as the parent, only the weights differ.",
            "x138 hard-codes the parent sha (code cell 3 _secondary_expected_sha256); set it to model.weight_sha256 and "
            "point BIOHUB_SECONDARY_ARTIFACT_MANIFEST at this file.",
        ],
    }
    (out_dir / "ARTIFACT_MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

    # Emulate x138 code cell 3 on the written folder.
    _info = json.loads((out_dir / "ARTIFACT_MANIFEST.json").read_text())
    assert str(_info.get("model", {}).get("weight_sha256", "")) == out_sha256
    with tempfile.TemporaryDirectory() as _tmp:
        _copied = Path(_tmp) / "secondary_seed_weights"
        shutil.copytree(out_dir / "weights", _copied)
        _w = _copied / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
        assert _w.is_file() and _w.with_name("config.json").is_file()
        assert hashlib.sha256(_w.read_bytes()).hexdigest() == out_sha256
        assert sorted(p.name for p in _w.parent.iterdir()) == ["config.json", "edge_predictor_best.pth"]
    print(f"[{label}] x138 loader emulation OK -> {out_dir}", flush=True)
    return {
        "dir": str(out_dir),
        "artifact_name": spec["artifact_name"],
        "weight_sha256": out_sha256,
        "parent_sha256": spec["init_sha256"],
        "root_parent_sha256": expected_initial_sha256,
        "n_iters": n_iters,
        "max_iters": train_stats.get("max_iters"),
        "hard_stopped": bool(train_stats.get("hard_stopped")),
        "stop_reason": train_stats.get("stop_reason"),
    }


for _label in [spec["label"] for spec in RUN_SPECS]:
    if _label not in SANITY:
        continue
    try:
        EXPORTED[_label] = _export(_label)
    except Exception as exc:
        traceback.print_exc()
        EXPORT_FAIL[_label] = repr(exc)
        shutil.rmtree(WORKING_ROOT / SPEC_BY_LABEL[_label]["export_dir"], ignore_errors=True)
        print(f"[exp77] {_label}: EXPORT FAILED: {exc!r}", flush=True)

EXP77_LOG["artifacts"] = EXPORTED
EXP77_LOG["export_failures"] = EXPORT_FAIL
EXP77_LOG["runs"] = {label: res.get("worker_log") for label, res in WORKER_RESULTS.items()}
EXP77_LOG["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
EXP77_LOG["notebook_wall_s"] = round(time.time() - EXP77_T0, 1)
(WORKING_ROOT / "exp77_train_log.json").write_text(json.dumps(EXP77_LOG, indent=2) + "\n", encoding="utf-8")

# Cleanup. run_dir holds the repo copy, the teacher copy, both students' scratch weights, splits.json and the run configs.
# The worker source moves next to its logs.
shutil.rmtree(run_dir)
if WORKER_SRC.exists():
    shutil.move(str(WORKER_SRC), str(EXP77_LOGS / WORKER_SRC.name))
_total = 0
print("final /kaggle/working:")
for _p in sorted(WORKING_ROOT.rglob("*")):
    if _p.is_file():
        _total += _p.stat().st_size
        print(f"  {_p.relative_to(WORKING_ROOT)}  {_p.stat().st_size:,} B")
print(f"  total {_total / 2**20:.1f} MiB")
if _total > 64 * 2**20:
    print("WARNING: /kaggle/working is larger than expected", flush=True)
for _label in [spec["label"] for spec in RUN_SPECS]:
    if _label in EXPORTED:
        _e = EXPORTED[_label]
        print(f"NEW SECONDARY {_label} ({_e['artifact_name']}): SHA256 {_e['weight_sha256']} | n_iters {_e['n_iters']} | "
              f"max_iters {_e['max_iters']} | hard_stopped {_e['hard_stopped']} | stop_reason {_e['stop_reason']}")
    else:
        _why = EXPORT_FAIL.get(_label) or SANITY_FAIL.get(_label) or WORKER_RESULTS.get(_label, {}).get("status")
        print(f"!!! {_label}: NOT EXPORTED ({_why}); see exp77_logs/worker_{_label}.log", flush=True)
if not EXPORTED:
    raise RuntimeError("exp77: no student exported")
